# Analyse des Synthetischen Regressors (Style-Score)

Dieses Notebook evaluiert den trainierten **Synthetischen Regressor**, der auf LLM-generierten Stufen ($0.25, 0.5, 0.75$) trainiert wurde.
1. Laden des Modells `bilstm_synthetic_regression.pt`.
2. Laden des Vokabulars aus `data/vocabs/synthetic_vocab.json`.
3. Vorhersagen auf dem Test-Split.
4. Evaluation von benutzerdefinierten Texten.


In [1]:
import os
import json
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import spacy
import matplotlib.pyplot as plt
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

while not os.path.exists(".git"):
    parent = os.path.dirname(os.getcwd())
    if parent == os.getcwd():
        break
    os.chdir("..")
print("Arbeitsverzeichnis:", os.getcwd())


Arbeitsverzeichnis: /Users/fietescheel/Documents/Master Thesis


In [2]:
VOCAB_PATH = "data/new_pipeline/vocabs/synthetic_vocab.json"
MODEL_PATH = "results/models/new_pipeline/bilstm_synthetic_regression.pt"
DEVICE = torch.device("cuda" if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu"))
print("Nutze Device:", DEVICE)


Nutze Device: mps


## 1. Vokabular & Modell laden


In [3]:
class BiLSTMRegressor(nn.Module):
    def __init__(self, vocab_size, embed_dim=128, hidden_dim=128, dropout=0.3):
        super(BiLSTMRegressor, self).__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.lstm = nn.LSTM(embed_dim, hidden_dim, batch_first=True, bidirectional=True)
        self.fc = nn.Linear(hidden_dim * 2, 1)
        self.dropout = nn.Dropout(dropout)
        self.sigmoid = nn.Sigmoid()
        
    def forward(self, x):
        embedded = self.dropout(self.embedding(x))
        _, (hidden, _) = self.lstm(embedded)
        hidden = torch.cat((hidden[-2,:,:], hidden[-1,:,:]), dim=1)
        out = self.fc(self.dropout(hidden))
        return self.sigmoid(out).squeeze(-1)

with open(VOCAB_PATH, "r", encoding="utf-8") as f:
    stoi = json.load(f)
print(f"Vokabular-Größe geladen: {len(stoi)}")

model = BiLSTMRegressor(len(stoi)).to(DEVICE)
if os.path.exists(MODEL_PATH):
    model.load_state_dict(torch.load(MODEL_PATH, map_location=DEVICE))
    print("Synthetisches Regressor-Modell geladen!")
else:
    print(f"WARNUNG: Modell-Gewichte unter {MODEL_PATH} nicht gefunden!")
model.eval()


Vokabular-Größe geladen: 47675
WARNUNG: Modell-Gewichte unter results/models/new_pipeline/bilstm_synthetic_regression.pt nicht gefunden!


BiLSTMRegressor(
  (embedding): Embedding(47675, 128, padding_idx=0)
  (lstm): LSTM(128, 128, batch_first=True, bidirectional=True)
  (fc): Linear(in_features=256, out_features=1, bias=True)
  (dropout): Dropout(p=0.3, inplace=False)
  (sigmoid): Sigmoid()
)

## 2. Eigene Sätze testen


In [4]:
nlp = spacy.load("de_core_news_sm", disable=["ner", "tagger", "lemmatizer"])

def get_simplicity_score(text):
    tokens = [t.text.lower() for t in nlp(text) if not t.is_space]
    encoded = [stoi.get(t, stoi.get("<unk>", 1)) for t in tokens]
    inp = torch.tensor([encoded], dtype=torch.long).to(DEVICE)
    
    with torch.no_grad():
        score = model(inp).item()
    return score

sents = [
    "Hier ist ein extrem komplexer Satz mit Schachtelsätzen und Fremdwörtern.",
    "Das ist ein einfacherer Satz mit wenigen Wörtern.",
    "Das ist Leichte Sprache."
]
for s in sents:
    print(f"Satz: '{s}' -> Score: {get_simplicity_score(s):.4f}")


Satz: 'Hier ist ein extrem komplexer Satz mit Schachtelsätzen und Fremdwörtern.' -> Score: 0.4970
Satz: 'Das ist ein einfacherer Satz mit wenigen Wörtern.' -> Score: 0.4791
Satz: 'Das ist Leichte Sprache.' -> Score: 0.4798
